# BnSentMix — Model 6 Manual Perturbation Audit Generator

## Purpose

This notebook creates the human-validation sample for the frozen Model 6 Romanization intervention. It does **not** train, load, select, or evaluate a sentiment model.

The study's robustness question remains unchanged:

> Does Romanization-consistency training reduce sensitivity to plausible Romanized spelling variation while preserving clean sentiment performance?

Before using robustness results in the paper, two independent Bengali-English readers must verify that the perturbations are meaning-preserving, label-preserving, natural, and not excessively artificial.

The notebook produces exactly **300 clean–perturbed pairs: 75 from each sentiment label**. The final test set remains sealed.

## Frozen protocol

| Item | Fixed value |
|---|---|
| Dataset | `/kaggle/input/datasets/olidopu/bnsentmix/dataset.csv` |
| Dataset SHA-256 | `148f23...029d` |
| Split | Same group-disjoint split used by Models 1–6 |
| Split seed | `2026` |
| Split SHA-256 | `7be81e...8ec5` |
| Audit source | Validation split only |
| Perturbation rules | Exact frozen Model 6 rules |
| Perturbation seed | `6006` |
| Variant | `0` |
| Sampling | 75 changed, unique-group pairs per label |
| Sampling seed | `6006 + label_id` |
| Final shuffle seed | `6006` |
| Final test | Not perturbed, sampled, or evaluated |

Label mapping was confirmed earlier and is not re-audited here: `0=positive`, `1=negative`, `2=neutral`, `3=mixed`.

This notebook runs on CPU. Internet, GPU, tokenizer, model weights, and checkpoints are not required.

In [ ]:
# Lightweight imports: no Transformers or GPU packages are needed.
from pathlib import Path
from difflib import SequenceMatcher
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import FileLink, display
from sklearn.model_selection import StratifiedGroupKFold


# Dataset and output paths.
DATA_PATH = Path('/kaggle/input/datasets/olidopu/bnsentmix/dataset.csv')
OUTPUT_DIR = Path('/kaggle/working/bnsentmix_manual_audit')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_CSV_PATH = OUTPUT_DIR / 'manual_perturbation_audit_balanced_300.csv'
MANIFEST_PATH = OUTPUT_DIR / 'manual_perturbation_audit_manifest.json'
LOCKED_SPLIT_PATH = OUTPUT_DIR / 'bnsentmix_locked_split.csv'


# Frozen dataset and split fingerprints from Models 1–6.
EXPECTED_RAW_SHA256 = '148f23eb3dc40c1012a973efec920eaccc39700a74e5bcfb56806b0bf389029d'
EXPECTED_SPLIT_SHA256 = '7be81e52030967319d43ddc1e1feaf9817f9c3011a9b914265c34b5ce0ab8ec5'
EXPECTED_SPLIT_COUNTS = {'train': 13862, 'validation': 2975, 'test': 2973}
SPLIT_SEED = 2026


# Confirmed label mapping; this notebook performs no new label audit.
LABELS = [0, 1, 2, 3]
LABEL_NAMES = {0: 'positive', 1: 'negative', 2: 'neutral', 3: 'mixed'}


# Exact frozen Model 6 Romanization parameters.
TOKEN_CHANGE_PROBABILITY = 0.35
MAX_TOKEN_CHANGES = 2
CLITIC_JOIN_PROBABILITY = 0.20
ROBUSTNESS_SEED = 6006
AUDIT_VARIANT_ID = 0


# Frozen manual-audit sampling plan.
SAMPLES_PER_LABEL = 75
TOTAL_AUDIT_SAMPLES = SAMPLES_PER_LABEL * len(LABELS)
# Sample the minority mixed class first when excluding repeated text groups.
SAMPLING_LABEL_ORDER = [3, 0, 1, 2]

print('Audit output directory:', OUTPUT_DIR)
print('Model training required: False')
print('Final test opened: False')

In [ ]:
# Reconstruct and verify the exact leakage-free split used by Models 1–6.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def normalized_group_key(text):
    text = unicodedata.normalize('NFKC', str(text)).lower()
    text = re.sub(r'[\u200b-\u200d\ufeff]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


raw = pd.read_csv(DATA_PATH)
assert sha256_file(DATA_PATH) == EXPECTED_RAW_SHA256, (
    'Raw dataset differs from the frozen Models 1–6 dataset.'
)

lookup = {column.strip().lower(): column for column in raw.columns}
text_col = next(
    lookup[name] for name in ['sentence', 'text', 'comment'] if name in lookup
)
label_col = next(
    lookup[name] for name in ['label', 'sentiment', 'class'] if name in lookup
)

df = raw[[text_col, label_col]].rename(
    columns={text_col: 'text', label_col: 'label'}
).copy()
df.insert(0, 'source_row_id', np.arange(len(df)))
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(int)
df = df[df['text'].ne('')].copy()
df['group_id'] = df['text'].map(normalized_group_key)
df = df.drop_duplicates(['group_id', 'label'], keep='first').reset_index(drop=True)

splitter = StratifiedGroupKFold(
    n_splits=20, shuffle=True, random_state=SPLIT_SEED
)
fold_ids = np.full(len(df), -1, dtype=int)
for fold_id, (_, held_out) in enumerate(
    splitter.split(df['text'], df['label'], groups=df['group_id'])
):
    fold_ids[held_out] = fold_id

df['split'] = np.where(
    fold_ids < 14,
    'train',
    np.where(fold_ids < 17, 'validation', 'test'),
)

assert df.groupby('group_id')['split'].nunique().max() == 1
assert df['split'].value_counts().to_dict() == EXPECTED_SPLIT_COUNTS

# Writing the split reproduces the original split fingerprint exactly.
df.to_csv(LOCKED_SPLIT_PATH, index=False)
assert sha256_file(LOCKED_SPLIT_PATH) == EXPECTED_SPLIT_SHA256, (
    'Reconstructed split differs from the frozen Models 1–6 split.'
)

validation_df = df[df['split'].eq('validation')].copy()
test_source_ids = set(df.loc[df['split'].eq('test'), 'source_row_id'])

display(pd.crosstab(df['split'], df['label']).rename(columns=LABEL_NAMES))
print('Verified dataset SHA-256:', EXPECTED_RAW_SHA256)
print('Verified split SHA-256:', EXPECTED_SPLIT_SHA256)
print('Validation rows available:', len(validation_df))

In [ ]:
# Exact conservative Model 6 Romanization candidate rules.
LATIN_WORD_REGEX = re.compile(r'[A-Za-z]+')
CLITIC_REGEX = re.compile(
    r'\b([A-Za-z]{2,})\s+(ta|ti|gula|gulo|er|ke|te)\b',
    flags=re.IGNORECASE,
)
DIGRAPH_ALTERNATIVES = (
    ('sh', 's'), ('ph', 'f'), ('bh', 'b'), ('kh', 'k'),
    ('gh', 'g'), ('ch', 'c'), ('th', 't'), ('dh', 'd'),
)
CONSONANT_ALTERNATIVES = (
    ('z', 'j'), ('j', 'z'), ('v', 'b'), ('b', 'v')
)


def stable_uint32(*parts):
    payload = '|'.join(map(str, parts)).encode('utf-8')
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], 'little')


def restore_case(original, changed_lower):
    if original.isupper():
        return changed_lower.upper()
    if original[:1].isupper() and original[1:].islower():
        return changed_lower.capitalize()
    return changed_lower


def spelling_candidates(token):
    lower = token.lower()
    candidates = {}

    # Aspiration/digraph reductions: bhalo→balo, kharap→karap, etc.
    for source, replacement in DIGRAPH_ALTERNATIVES:
        start = lower.find(source)
        if start >= 0:
            changed = lower[:start] + replacement + lower[start + len(source):]
            candidates[changed] = 'digraph_reduction'

    # Informal consonant alternatives.
    for source, replacement in CONSONANT_ALTERNATIVES:
        start = lower.find(source)
        if start >= 0:
            changed = lower[:start] + replacement + lower[start + 1:]
            candidates[changed] = 'consonant_alternative'

    # Collapse an existing repeated vowel.
    repeated = re.search(r'([aeiou])\1+', lower)
    if repeated:
        start, end = repeated.span()
        changed = lower[:start] + repeated.group(1) + lower[end:]
        candidates[changed] = 'vowel_collapse'

    internal_vowels = [
        index for index, char in enumerate(lower)
        if char in 'aeiou' and 0 < index < len(lower) - 1
    ]
    if internal_vowels:
        index = internal_vowels[0]
        candidates[lower[:index] + lower[index] + lower[index:]] = 'vowel_repeat'
        if len(lower) >= 5:
            candidates[lower[:index] + lower[index + 1:]] = 'vowel_omission'

    # Short tokens often have no internal vowel; repeat one existing vowel.
    if not candidates:
        vowel_positions = [
            index for index, char in enumerate(lower) if char in 'aeiou'
        ]
        if vowel_positions:
            index = vowel_positions[0]
            candidates[
                lower[:index] + lower[index] + lower[index:]
            ] = 'vowel_repeat'

    candidates.pop(lower, None)
    return [
        (restore_case(token, changed), operation)
        for changed, operation in sorted(candidates.items())
    ]

In [ ]:
# Exact deterministic perturbation procedure from Model 6.
def romanization_variant(text, random_seed):
    text = str(text)
    rng = np.random.default_rng(random_seed)
    matches = list(LATIN_WORD_REGEX.finditer(text))
    eligible = []

    for match in matches:
        token = match.group(0)
        if len(token) >= 3:
            candidates = spelling_candidates(token)
            if candidates:
                eligible.append((match, candidates))

    selected = []
    if eligible:
        order = rng.permutation(len(eligible))
        for position in order:
            if rng.random() < TOKEN_CHANGE_PROBABILITY:
                selected.append(eligible[int(position)])
            if len(selected) == MAX_TOKEN_CHANGES:
                break
        if not selected:
            selected = [eligible[int(order[0])]]

    replacements = {}
    operations = []
    for match, candidates in selected:
        replacement, operation = candidates[int(rng.integers(len(candidates)))]
        replacements[(match.start(), match.end())] = replacement
        operations.append(operation)

    pieces = []
    cursor = 0
    for match in matches:
        pieces.append(text[cursor:match.start()])
        pieces.append(
            replacements.get((match.start(), match.end()), match.group(0))
        )
        cursor = match.end()
    pieces.append(text[cursor:])
    variant = ''.join(pieces)

    # Join an explicit Bangla clitic without deleting any characters.
    clitic_matches = list(CLITIC_REGEX.finditer(variant))
    if clitic_matches and rng.random() < CLITIC_JOIN_PROBABILITY:
        match = clitic_matches[int(rng.integers(len(clitic_matches)))]
        variant = (
            variant[:match.start()]
            + match.group(1)
            + match.group(2)
            + variant[match.end():]
        )
        operations.append('clitic_join')

    return variant, operations


def build_variant_frame(frame, base_seed, variant_id):
    rows = []
    required_columns = ['source_row_id', 'group_id', 'text', 'label']

    for row in frame[required_columns].itertuples(index=False):
        row_seed = stable_uint32(base_seed, row.source_row_id, variant_id)
        variant_text, operations = romanization_variant(row.text, row_seed)
        rows.append({
            'source_row_id': row.source_row_id,
            'group_id': row.group_id,
            'label': row.label,
            'original_text': row.text,
            'variant_text': variant_text,
            'operations': '|'.join(operations) if operations else 'unchanged',
            'changed': variant_text != row.text,
            'character_change_ratio': 1.0 - SequenceMatcher(
                None, row.text, variant_text
            ).ratio(),
        })

    return pd.DataFrame(rows)

In [ ]:
# Generate only frozen validation variant 0. The test split is never perturbed.
validation_variant_0 = build_variant_frame(
    validation_df,
    base_seed=ROBUSTNESS_SEED,
    variant_id=AUDIT_VARIANT_ID,
)

candidate_summary = validation_variant_0.groupby('label').agg(
    total_pairs=('source_row_id', 'size'),
    changed_pairs=('changed', 'sum'),
    changed_rate=('changed', 'mean'),
    mean_character_change=('character_change_ratio', 'mean'),
).rename(index=LABEL_NAMES)

display(candidate_summary.round(4))
print('Candidate source split: validation only')
print('Test examples perturbed: 0')

In [ ]:
# Select 75 changed pairs per label with no repeated source or normalized text group.
changed_candidates = validation_variant_0[
    validation_variant_0['changed']
].copy()

selected_parts = []
used_group_ids = set()

for label_id in SAMPLING_LABEL_ORDER:
    label_pool = changed_candidates[
        changed_candidates['label'].eq(label_id)
        & ~changed_candidates['group_id'].isin(used_group_ids)
    ]

    if len(label_pool) < SAMPLES_PER_LABEL:
        raise RuntimeError(
            f'Label {label_id} has only {len(label_pool)} eligible unique-group pairs.'
        )

    label_sample = label_pool.sample(
        n=SAMPLES_PER_LABEL,
        replace=False,
        random_state=ROBUSTNESS_SEED + label_id,
    )
    selected_parts.append(label_sample)
    used_group_ids.update(label_sample['group_id'])


audit_sample = pd.concat(selected_parts, ignore_index=True)
audit_sample = audit_sample.sample(
    frac=1,
    random_state=ROBUSTNESS_SEED,
).reset_index(drop=True)

audit_sample.insert(
    0,
    'audit_id',
    [f'AUDIT_{index:03d}' for index in range(1, len(audit_sample) + 1)],
)
audit_sample.insert(
    4,
    'label_name',
    audit_sample['label'].map(LABEL_NAMES),
)

# Protocol checks must all pass before saving.
expected_label_counts = {label: SAMPLES_PER_LABEL for label in LABELS}
actual_label_counts = (
    audit_sample['label'].value_counts().sort_index().to_dict()
)

assert len(audit_sample) == TOTAL_AUDIT_SAMPLES
assert actual_label_counts == expected_label_counts
assert audit_sample['audit_id'].nunique() == TOTAL_AUDIT_SAMPLES
assert audit_sample['source_row_id'].nunique() == TOTAL_AUDIT_SAMPLES
assert audit_sample['group_id'].nunique() == TOTAL_AUDIT_SAMPLES
assert audit_sample['changed'].all()
assert not set(audit_sample['source_row_id']).intersection(test_source_ids)

display(
    audit_sample['label_name']
    .value_counts()
    .reindex([LABEL_NAMES[label] for label in LABELS])
    .rename('audit_pairs')
    .to_frame()
)
display(audit_sample[[
    'audit_id', 'label_name', 'original_text', 'variant_text', 'operations'
]].head(10))

print('Balanced audit checks: PASS')
print('Unique source rows:', audit_sample['source_row_id'].nunique())
print('Unique normalized groups:', audit_sample['group_id'].nunique())
print('Final test examples included:', 0)

In [ ]:
# Save the balanced master CSV and its reproducibility manifest.
output_columns = [
    'audit_id', 'source_row_id', 'group_id', 'label', 'label_name',
    'original_text', 'variant_text', 'operations', 'changed',
    'character_change_ratio',
]
audit_sample[output_columns].to_csv(AUDIT_CSV_PATH, index=False)
audit_csv_sha256 = sha256_file(AUDIT_CSV_PATH)

manifest = {
    'purpose': 'Manual validity audit for frozen Model 6 Romanization perturbations',
    'research_question': (
        'Does Romanization-consistency training reduce spelling-variation '
        'sensitivity while preserving clean sentiment performance?'
    ),
    'dataset_path': str(DATA_PATH),
    'dataset_sha256': EXPECTED_RAW_SHA256,
    'split_seed': SPLIT_SEED,
    'split_sha256': EXPECTED_SPLIT_SHA256,
    'source_split': 'validation',
    'test_opened': False,
    'perturbation_variant_id': AUDIT_VARIANT_ID,
    'robustness_seed': ROBUSTNESS_SEED,
    'token_change_probability': TOKEN_CHANGE_PROBABILITY,
    'max_token_changes': MAX_TOKEN_CHANGES,
    'clitic_join_probability': CLITIC_JOIN_PROBABILITY,
    'samples_per_label': SAMPLES_PER_LABEL,
    'total_samples': TOTAL_AUDIT_SAMPLES,
    'sampling_label_order': SAMPLING_LABEL_ORDER,
    'sample_only_changed_pairs': True,
    'unique_source_rows': int(audit_sample['source_row_id'].nunique()),
    'unique_group_ids': int(audit_sample['group_id'].nunique()),
    'label_counts': {
        str(label): int(count)
        for label, count in actual_label_counts.items()
    },
    'audit_csv_sha256': audit_csv_sha256,
}

with open(MANIFEST_PATH, 'w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2, ensure_ascii=False)

print('Audit CSV SHA-256:', audit_csv_sha256)
print('Saved audit CSV:', AUDIT_CSV_PATH)
print('Saved manifest:', MANIFEST_PATH)

display(FileLink(str(AUDIT_CSV_PATH)))
display(FileLink(str(MANIFEST_PATH)))

## After running

Upload `manual_perturbation_audit_balanced_300.csv` here. Two blinded reviewer sheets will then be prepared from the master sample. Reviewers will not need model predictions or checkpoints.

Do not manually change, reorder, remove, or replace audit rows after generation. The manifest records the sample's SHA-256 fingerprint for reproducibility.